
# 🛍️ PRACTICAL: BazarKaap Site — 7 Shop Tickets + BOSS! 👑

**Folder:** 00_Python → 13_Other_Libraries → 03_Flask_WebDev
**Type:** PRACTICAL · **Data:** 4-product catalog (self-contained!)

---
## 🎮 Story
BazarKaap's website is in your hands! The owner's 7 tickets: catalog page, product
page, cart-add (Form!), filter (?max_price=), checkout (bill+empty!), low-stock
alerts, 404-page — and at the end the BOSS: 10 asserts → **SHOPKEEPER Badge**! 🏆

**Grading:** ✅ EASY (1-2) · 🟡 MEDIUM (3-5) · 🔴 HARD (6-7) + 👑 BOSS

<details><summary>📦 <b>INTAKE (click)</b></summary>


In [1]:
# 🧺 INTAKE
from flask import Flask, jsonify, request, redirect, url_for, render_template_string

PRODUCTS = [
    {"id": 1, "name": "Pen",      "price": 20,  "stock": 50},
    {"id": 2, "name": "Notebook", "price": 50,  "stock": 30},
    {"id": 3, "name": "Bag",      "price": 600, "stock": 5},
    {"id": 4, "name": "Bottle",   "price": 90,  "stock": 12},
]
CART = []
print("catalog ready:", [(p["name"], p["price"]) for p in PRODUCTS])
# Expected OUTPUT:
# catalog ready: [('Pen', 20), ('Notebook', 50), ('Bag', 600), ('Bottle', 90)]

catalog ready: [('Pen', 20), ('Notebook', 50), ('Bag', 600), ('Bottle', 90)]


</details>

---
## ✅ Ticket 1 (EASY): Catalog JSON 📋
`GET /api/catalog` → {"count":4, "items":[...]}

In [2]:
# ✅ S1 TODO:


## ✅ Ticket 2 (EASY): Single-Product HTML Page 📄
`GET /product/<int:pid>` → HTML with the name + price (if not found → 404-tuple!)


In [3]:
# ✅ S2 TODO:

---
## ✅ Ticket 3 (MEDIUM): Cart Add (POST-Form!) 🛒
`POST /cart/add` (form: pid, qty) → CART.append → redirect /cart ; bad pid → 400

In [4]:
# ✅ S3 TODO:


## ✅ Ticket 4 (MEDIUM): Cart Page (Jinja Loop!) 🎨
`GET /cart` → a ul-loop in render_template_string — item names in the HTML!


In [5]:
# ✅ S4 TODO:


## ✅ Ticket 5 (MEDIUM): Price Filter 🔎
`GET /catalog?max_price=100` → JSON with only the items <=100!


In [6]:
# ✅ S5 TODO:


## ✅ Ticket 6 (HARD): Checkout! 🧾
`POST /checkout` → total-count, CART emptied; empty-cart → 400!


In [7]:
# ✅ S6 TODO:


## ✅ Ticket 7 (HARD): Low-Stock Alerts Page ⚠️
`GET /alerts` → Jinja if: stock<10 then a warn-list!


In [8]:
# ✅ S7 TODO:

## 👑 BOSS: 10-Assert Shopkeeper Badge (a fresh shop = self-contained!) 🏆

In [9]:
# 👑 BOSS CONSOLE
_b_app = Flask("boss-dukaan")
_b_products = [dict(p) for p in PRODUCTS]
_b_cart = []

@_b_app.get("/api/catalog")
def _catalog():
    return jsonify({"count": len(_b_products), "items": _b_products})

@_b_app.get("/product/<int:pid>")
def _prod(pid):
    for p in _b_products:
        if p["id"] == pid:
            return f"<h2>{p['name']}</h2><p>₹{p['price']}</p>"
    return jsonify({"error": "nahi-mila"}), 404

@_b_app.post("/cart/add")
def _add():
    pid = int(request.form["pid"]); qty = int(request.form.get("qty", 1))
    for p in _b_products:
        if p["id"] == pid:
            _b_cart.append({"name": p["name"], "qty": qty, "amt": p["price"] * qty})
            return redirect(url_for("_cart_page"))
    return jsonify({"error": "bekaar pid"}), 400

@_b_app.get("/cart")
def _cart_page():
    return render_template_string(
        "<ul>{% for i in c %}<li>{{ i.name }} x{{ i.qty }} = ₹{{ i.amt }}</li>{% endfor %}</ul>",
        c=_b_cart)

@_b_app.get("/catalog")
def _filter():
    mx = int(request.args.get("max_price", "10**9"))
    return jsonify([p["name"] for p in _b_products if p["price"] <= mx])

@_b_app.post("/checkout")
def _checkout():
    if not _b_cart:
        return jsonify({"error": "cart khaali!"}), 400
    total = sum(i["amt"] for i in _b_cart)
    n = len(_b_cart)
    _b_cart.clear()
    return jsonify({"items": n, "total": total})

@_b_app.errorhandler(404)
def _404(e):
    return "<h1>dukaan me ye dabba nahi!</h1>", 404

c = _b_app.test_client()
assert c.get("/api/catalog").get_json()["count"] == 4,                          "① catalog 4!"
assert "Bag" in c.get("/product/3").data.decode() and "600" in c.get("/product/3").data.decode(), "② Bag page!"
assert c.get("/product/9").status_code == 404,                              "③ pid-9 → 404!"
c.post("/cart/add", data={"pid": "2", "qty": "2"})
c.post("/cart/add", data={"pid": "4"})
assert len(_b_cart) == 2 and _b_cart[0] == {"name": "Notebook", "qty": 2, "amt": 100}, "④ cart math (50*2)!"
assert c.post("/cart/add", data={"pid": "77"}).status_code == 400,          "⑤ bekaar pid → 400!"
cart_html = c.get("/cart").data.decode()
assert "Notebook x2 = ₹100" in cart_html and "Bottle x1 = ₹90" in cart_html, "⑥ cart page jinja-loop!"
assert c.get("/catalog?max_price=100").get_json() == ["Pen", "Notebook", "Bottle"], "⑦ filter <=100!"
r_pay = c.post("/checkout")
assert r_pay.get_json() == {"items": 2, "total": 190},                          "⑧ bill = 100+90 = 190!"
assert _b_cart == [] and c.post("/checkout").status_code == 400,            "⑨ empty-checkout → 400!"
assert "dukaan me ye dabba nahi" in c.get("/abcxyz").data.decode(),         "⑩ custom-404 page!"
print("🏆 10/10 SHOPKEEPER BADGE — website tumhare routes pe chalti! 🛍️👑")

🏆 10/10 SHOPKEEPER BADGE — website tumhare routes pe chalti! 🛍️👑


---
## ✅ SOLUTIONS 🔓

<details><summary><b>S1-S4</b></summary>

In [10]:
# S1-S4 ✅
app = Flask("bazarkaap-main")

@app.get("/api/catalog")
def catalog():
    return jsonify({"count": len(PRODUCTS), "items": PRODUCTS})

@app.get("/product/<int:pid>")
def product(pid):
    for p in PRODUCTS:
        if p["id"] == pid:
            return f"<h2>{p['name']}</h2><p>₹{p['price']}</p>"
    return jsonify({"error": f"product-{pid} nahi"}), 404

@app.post("/cart/add")
def cart_add():
    pid = int(request.form["pid"]); qty = int(request.form.get("qty", 1))
    for p in PRODUCTS:
        if p["id"] == pid:
            CART.append({"name": p["name"], "qty": qty, "amt": p["price"] * qty})
            return redirect(url_for("cart_page"))
    return jsonify({"error": "bekaar pid"}), 400

@app.get("/cart")
def cart_page():
    return render_template_string(
        "<ul>{% for i in c %}<li>{{ i.name }} x{{ i.qty }} = ₹{{ i.amt }}</li>{% endfor %}</ul>",
        c=CART)

c1 = app.test_client()
print("①", c1.get("/api/catalog").get_json()["count"], "items")
print("②", c1.get("/product/1").data.decode(), "| miss:", c1.get("/product/9").status_code)
c1.post("/cart/add", data={"pid": "1", "qty": "3"}); c1.post("/cart/add", data={"pid": "2", "qty": "1"})
print("③ cart-len:", len(CART), "| bad:", c1.post("/cart/add", data={"pid": "77"}).status_code)
print("④", c1.get("/cart").data.decode())

① 4 items
② <h2>Pen</h2><p>₹20</p> | miss: 404
③ cart-len: 2 | bad: 400
④ <ul><li>Pen x3 = ₹60</li><li>Notebook x1 = ₹50</li></ul>


<details><summary><b>S5-S7</b></summary>

In [11]:
# S5-S7 ✅  (dev-restart-roop: naya app — pichhla freeze the!)
app_b = Flask("bazarkaap-v2")

@app_b.get("/catalog")
def filt():
    mx = int(request.args.get("max_price", "10**9"))
    return jsonify([p["name"] for p in PRODUCTS if p["price"] <= mx])

@app_b.post("/checkout")
def checkout():
    if not CART:
        return jsonify({"error": "cart khaali!"}), 400
    total, n = sum(i["amt"] for i in CART), len(CART)
    CART.clear()
    return jsonify({"items": n, "total": total})

@app_b.get("/alerts")
def alerts():
    return render_template_string(
        "<ul>{% for p in items %}{% if p.stock < 10 %}<li>⚠️ {{ p.name }} — {{ p.stock }}!</li>{% endif %}{% endfor %}</ul>",
        items=PRODUCTS)

c2 = app_b.test_client()
print("⑤ <=100:", c2.get("/catalog?max_price=100").get_json())
print("⑥ checkout:", c2.post("/checkout").get_json(), "| again:", c2.post("/checkout").status_code)
print("⑦ alerts:", c2.get("/alerts").data.decode())
# Takeaway: pages → forms → cart → checkout — poora dukaan-flow jhatt-khadi! 🛒

⑤ <=100: ['Pen', 'Notebook', 'Bottle']
⑥ checkout: {'items': 2, 'total': 110} | again: 400
⑦ alerts: <ul><li>⚠️ Bag — 5!</li></ul>



---
## 🏁 Checklist — Shopkeeper 🛍️
| ✔ | Skill |
|---|---|
| ⬜ | flask route + int-slot 404-police 👮 |
| ⬜ | request.form / request.args extract (int-cast!) |
| ⬜ | status-tuple + jsonify (body, 404) |
| ⬜ | redirect(url_for(...)) after POST |
| ⬜ | render_template_string loops/ifs 🎨 |

**ALL GREEN?** 🎉 Next: **04_Streamlit_Dashboards** — a DATA-APP with NO web-code:
a Zomato-explorer dashboard in 5 lines! 📊⚡ (a star-topic! ⭐)
